# Game Translation A/B/C Experiment Runner

A/B/C 프롬프트 실험 자동 실행 노트북입니다.

- A: worldview only
- B: glossary + rules
- C: glossary + rules + sentence type


In [ ]:
# If needed:
# %pip install -q openai pandas python-dotenv tqdm


In [ ]:
import os
import re
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

try:
    from dotenv import load_dotenv
except ImportError:
    load_dotenv = None

try:
    from openai import OpenAI
except ImportError as e:
    raise ImportError("openai 패키지가 없습니다. 위 설치 셀을 먼저 실행하세요.") from e


In [ ]:
# =====================
# Config
# =====================
BASE_DIR = Path('/Users/yiji/Desktop/work/pseudo_lab/game_translation_exp')
DATA_DIR = BASE_DIR / 'data'
PROMPT_DIR = BASE_DIR / 'prompts'
EVAL_DIR = BASE_DIR / 'eval'

# .env loading
if load_dotenv is not None:
    load_dotenv(BASE_DIR / '.env', override=False)
    load_dotenv(override=False)

SAMPLES_PATH = DATA_DIR / 'samples.csv'
GLOSSARY_PATH = DATA_DIR / 'glossary.csv'
STYLE_GUIDE_PATH = DATA_DIR / 'style_guide.md'

WORLDVIEW_CONTEXT = """
Destiny-like sci-fantasy universe.
Tone: mysterious, heroic, high-stakes, mythic-technical blend.
UI text should be concise; gameplay mechanics must remain precise.
""".strip()

SRC_LANG = 'English'
TGT_LANG = 'Korean'

MODEL = os.getenv('OPENAI_MODEL', 'gpt-5.4-mini')
TEMPERATURE = 0.1
MAX_OUTPUT_TOKENS = 220
DRY_RUN = False

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
if not DRY_RUN and not OPENAI_API_KEY:
    raise ValueError('OPENAI_API_KEY 환경변수를 설정해주세요 (.env 가능).')

RUN_DATE = datetime.now().strftime('%Y-%m-%d')
RUN_DIR = BASE_DIR / 'outputs' / f'run_{RUN_DATE}'
RUN_DIR.mkdir(parents=True, exist_ok=True)

print('RUN_DIR =', RUN_DIR)
print('MODEL =', MODEL)
print('DRY_RUN =', DRY_RUN)
print('OPENAI_API_KEY loaded =', bool(OPENAI_API_KEY))


In [ ]:
# Load templates/data
TEMPLATE_A = (PROMPT_DIR / 'A_worldview.txt').read_text(encoding='utf-8')
TEMPLATE_B = (PROMPT_DIR / 'B_glossary_rules.txt').read_text(encoding='utf-8')
TEMPLATE_C = (PROMPT_DIR / 'C_glossary_rules_type.txt').read_text(encoding='utf-8')

samples_df = pd.read_csv(SAMPLES_PATH)
glossary_df = pd.read_csv(GLOSSARY_PATH)
style_guide = STYLE_GUIDE_PATH.read_text(encoding='utf-8')

print('samples:', len(samples_df))
print(samples_df['sentence_type'].value_counts(dropna=False))
print('glossary entries:', len(glossary_df))


In [ ]:
def glossary_to_text(df: pd.DataFrame) -> str:
    lines = []
    for _, r in df.iterrows():
        src = str(r.get('source_term', '')).strip()
        tgt = str(r.get('target_term', '')).strip()
        note = str(r.get('note', '')).strip()
        if src and tgt:
            lines.append(f'- {src} => {tgt}' + (f' ({note})' if note and note.lower() != 'nan' else ''))
    return '
'.join(lines)

GLOSSARY_TEXT = glossary_to_text(glossary_df)


def build_prompt(condition: str, source_text: str, sentence_type: str) -> str:
    mapping = {
        '{SRC_LANG}': SRC_LANG,
        '{TGT_LANG}': TGT_LANG,
        '{SOURCE_TEXT}': source_text,
        '{WORLDVIEW_CONTEXT}': WORLDVIEW_CONTEXT,
        '{GLOSSARY}': GLOSSARY_TEXT,
        '{STYLE_GUIDE_AND_RULES}': style_guide,
        '{SENTENCE_TYPE}': sentence_type,
    }
    template = TEMPLATE_A if condition == 'A' else TEMPLATE_B if condition == 'B' else TEMPLATE_C
    out = template
    for k, v in mapping.items():
        out = out.replace(k, v)
    return out


def call_model(prompt: str) -> str:
    if DRY_RUN:
        return '[DRY_RUN]'
    client = OpenAI(api_key=OPENAI_API_KEY)
    resp = client.responses.create(
        model=MODEL,
        input=[
            {'role': 'system', 'content': 'You are a careful game localization assistant. Output translation only.'},
            {'role': 'user', 'content': prompt},
        ],
        temperature=TEMPERATURE,
        max_output_tokens=MAX_OUTPUT_TOKENS,
    )
    return re.sub(r'\s+', ' ', (resp.output_text or '').strip())


In [ ]:
def run_condition(condition: str, df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, r in tqdm(df.iterrows(), total=len(df), desc=f'Run {condition}'):
        sid = r['sample_id']
        src = str(r['source_text'])
        stype = str(r.get('sentence_type', ''))
        out = call_model(build_prompt(condition, src, stype))
        rows.append({'sample_id': sid, 'sentence_type': stype, 'source_text': src, 'translation': out})
        if not DRY_RUN:
            time.sleep(0.15)
    return pd.DataFrame(rows)

A_df = run_condition('A', samples_df)
B_df = run_condition('B', samples_df)
C_df = run_condition('C', samples_df)

(A_df[['sample_id','source_text','translation']]).to_csv(RUN_DIR / 'A_outputs.csv', index=False, encoding='utf-8')
(B_df[['sample_id','source_text','translation']]).to_csv(RUN_DIR / 'B_outputs.csv', index=False, encoding='utf-8')
(C_df[['sample_id','source_text','translation']]).to_csv(RUN_DIR / 'C_outputs.csv', index=False, encoding='utf-8')
print('Saved A/B/C outputs to', RUN_DIR)


In [ ]:
merged = samples_df[['sample_id','sentence_type','source_text']].copy()
merged = merged.merge(A_df[['sample_id','translation']].rename(columns={'translation':'condition_A'}), on='sample_id', how='left')
merged = merged.merge(B_df[['sample_id','translation']].rename(columns={'translation':'condition_B'}), on='sample_id', how='left')
merged = merged.merge(C_df[['sample_id','translation']].rename(columns={'translation':'condition_C'}), on='sample_id', how='left')

for col in [
    'A_meaning','A_term','A_consistency','A_naturalness','A_ui_fit',
    'B_meaning','B_term','B_consistency','B_naturalness','B_ui_fit',
    'C_meaning','C_term','C_consistency','C_naturalness','C_ui_fit',
    'best','error_type','notes'
]:
    merged[col] = ''

eval_path = EVAL_DIR / f'eval_sheet_prefilled_{RUN_DATE}.csv'
merged.to_csv(eval_path, index=False, encoding='utf-8')
print('Saved:', eval_path)
